# 04 · Detector YOLO — entrenamiento en Google Colab (GPU)

Este notebook entrena los **dos detectores** del proyecto con la GPU gratuita de Colab:

| Variante | Dataset | Clases | Para qué |
|---|---|---|---|
| **A** `1clase` | `detector_1clase` | 1 (`producto`) | encontrar cada bebida; la marca la decide después el clasificador CNN sobre el recorte |
| **B** `multiclase` | `multiclase` | 8 (`config/clases.json`) | línea base: el YOLO detecta y clasifica a la vez |

**Por qué Colab:** en la notebook (Ryzen 7 7735HS, sin GPU NVIDIA) una época del detector de 1 clase tarda
decenas de minutos (ver `docs/detector.md`, tiempos medidos con `scripts/detector_entrenar.py --prueba`);
el entrenamiento completo serían días de CPU. Con una T4 es cuestión de horas.

## Antes de empezar
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.
2. En la PC, en la raíz del repo: `python scripts/detector_empaquetar_colab.py`
   → genera `runs/detector/colab/inventario_detector.zip`.
3. Subí ese zip a Google Drive en **`MyDrive/inventario/inventario_detector.zip`**
   (o elegí `ORIGEN = 'subir'` más abajo para subirlo directo, más lento).

## Al terminar se descarga `resultados_detector.zip` con:
| Archivo del zip | Dónde va en la PC (repo principal) |
|---|---|
| `detector_1clase.pt`, `detector_multiclase.pt` | `modelos/detector/` |
| `runs/1clase/`, `runs/multiclase/` (results.csv, gráficos, eventos TensorBoard, best.pt) | `runs/detector/colab_1clase/`, `runs/detector/colab_multiclase/` |
| `metricas_colab.json`, `conteo_test_*.csv` | `runs/detector/colab/` |

También queda una copia en `MyDrive/inventario/resultados_detector.zip`.

> Todas las celdas de código son Python puro (sin `!` ni `%`): los comandos de consola se corren con `subprocess`.

In [ ]:
# 1. Entorno: intérprete, GPU e instalación de Ultralytics (misma versión que en la PC)
import os, sys, subprocess, time, json, shutil, zipfile
from pathlib import Path

print('Python:', sys.executable)
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'SIN GPU: cambiá el tipo de entorno')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.178'], check=True)

import torch, ultralytics
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución -> Cambiar tipo -> T4 GPU'

In [ ]:
# 2. Traer los datos: desde Drive (recomendado) o subiendo el zip a mano
ORIGEN = 'drive'          # 'drive' | 'subir'
CARPETA_DRIVE = Path('/content/drive/MyDrive/inventario')
ZIP_LOCAL = Path('/content/inventario_detector.zip')

if ORIGEN == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    t0 = time.time()
    shutil.copy(CARPETA_DRIVE / 'inventario_detector.zip', ZIP_LOCAL)   # copiar antes de descomprimir: mucho más rápido
    print(f'Copiado desde Drive en {time.time() - t0:.0f} s')
else:
    from google.colab import files
    subido = files.upload()
    Path(next(iter(subido))).rename(ZIP_LOCAL)

with zipfile.ZipFile(ZIP_LOCAL) as z:
    z.extractall('/content')
RAIZ = Path('/content/inventario_detector')
print((RAIZ / 'LEEME.txt').read_text())
for v in ('detector_1clase', 'multiclase'):
    for s in ('train', 'val', 'test'):
        print(v, s, len(list((RAIZ / v / 'images' / s).glob('*'))), 'imágenes')

## 3. Hiperparámetros y aumento de datos

Son **los mismos** que `HIPER` y `AUMENTO` de `scripts/detector_entrenar.py` (si cambiás uno, cambiá el otro).

- **Transfer learning:** se parte de `yolo26n.pt`, entrenado en COCO (80 clases, incluye `bottle` y `cup`).
  El *backbone* ya sabe extraer bordes, texturas y formas; solo hay que ajustar (fine-tuning) toda la red
  a nuestras bebidas y reemplazar la cabeza de clasificación (80 → 1 u 8 clases). Converge en muchas menos
  épocas y con muchos menos datos que entrenar desde cero.
- **Aumento:** `mosaic=1` (4 fotos en una: más objetos chicos y contextos), `hsv_*` (iluminación),
  `scale`/`translate` (distancia y encuadre), `flipud=0` siempre (en un estante nada está cabeza abajo).
  `fliplr=0.5` en **1clase** (una botella espejada sigue siendo "producto") y `fliplr=0` en **multiclase**
  (el texto y el logo espejados no existen en la realidad y el modelo podría aprender patrones falsos de marca).
- **`MAX_POR_ORIGEN`:** el dataset `darians_bottle` aporta ~15 000 fotos de 416 px con **una** botella cada una,
  muy distintas de un estante. Con el tope se toman como mucho N fotos de cada origen: épocas ~2-3 veces más
  cortas y un train más balanceado. `None` = usar todo.

In [ ]:
# 3. Configuración (igual que scripts/detector_entrenar.py)
HIPER = dict(epochs=60, patience=15, imgsz=640, batch=32, optimizer='auto', cos_lr=False,
             seed=0, deterministic=False, plots=True)          # batch 32 entra holgado en una T4 (16 GB) con yolo26n
AUMENTO_COMUN = dict(flipud=0.0, mosaic=1.0, close_mosaic=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                     degrees=0.0, translate=0.1, scale=0.5, mixup=0.0)
AUMENTO = {'1clase': dict(AUMENTO_COMUN, fliplr=0.5), 'multiclase': dict(AUMENTO_COMUN, fliplr=0.0)}
CARPETA = {'1clase': 'detector_1clase', 'multiclase': 'multiclase'}
EPOCAS = {'1clase': 60, 'multiclase': 80}     # con early stopping (patience=15) suele cortar antes
MAX_POR_ORIGEN = {'1clase': 4000, 'multiclase': None}
MODELO_INICIAL = 'yolo26n.pt'                 # 'yolo26s.pt' si se empaquetó con --con-s (más lento, más preciso)
RUNS = Path('/content/runs')

from ultralytics import YOLO, settings
settings.update({'tensorboard': True})        # Ultralytics escribe eventos de TensorBoard en cada corrida

In [ ]:
# 4. TensorBoard dentro de Colab (se actualiza mientras entrena: pestaña SCALARS)
ip = get_ipython()
ip.run_line_magic('load_ext', 'tensorboard')
ip.run_line_magic('tensorboard', f'--logdir {RUNS}')

In [ ]:
# 5. Funciones de entrenamiento
import random

def data_yaml_con_tope(variante):
    '''data.yaml de la variante; si hay tope por origen, escribe una lista train_tope.txt.'''
    import yaml
    base = RAIZ / CARPETA[variante]
    tope = MAX_POR_ORIGEN[variante]
    if not tope:
        return base / 'data.yaml'
    cfg = yaml.safe_load((base / 'data.yaml').read_text())
    imgs = sorted((base / 'images' / 'train').glob('*'))
    por_origen = {}
    for p in imgs:
        por_origen.setdefault(p.name.split('__')[0], []).append(p)
    rng = random.Random(0)
    elegidas = []
    for origen, grupo in sorted(por_origen.items()):
        sel = rng.sample(grupo, tope) if len(grupo) > tope else grupo
        print(f'  {origen}: {len(sel)} de {len(grupo)}')
        elegidas += sel
    lista = base / 'train_tope.txt'
    lista.write_text('\n'.join(str(p) for p in sorted(elegidas)) + '\n')
    extra = cfg['train'][1:] if isinstance(cfg['train'], list) else []   # sintéticos, si se empaquetaron
    cfg['train'] = [str(lista)] + [str((base / e).resolve()) for e in extra]
    cfg['path'] = str(base)
    ruta = base / 'data_tope.yaml'
    ruta.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
    return ruta

def entrenar(variante):
    data = data_yaml_con_tope(variante)
    pesos = RAIZ / MODELO_INICIAL if (RAIZ / MODELO_INICIAL).exists() else MODELO_INICIAL
    modelo = YOLO(str(pesos))
    params = dict(HIPER, **AUMENTO[variante])
    params.update(data=str(data), epochs=EPOCAS[variante], device=0, workers=2, cache=False,
                  single_cls=(variante == '1clase'), project=str(RUNS), name=variante, exist_ok=True)
    print(params)
    t0 = time.time()
    modelo.train(**params)
    minutos = (time.time() - t0) / 60
    carpeta = Path(modelo.trainer.save_dir)
    (carpeta / 'tiempo_colab.json').write_text(json.dumps(
        {'minutos': round(minutos, 1), 'gpu': torch.cuda.get_device_name(0), 'params': params}, indent=2))
    print(f'{variante}: {minutos:.1f} min -> {carpeta}')
    return carpeta

In [ ]:
# 6. Entrenar la variante A (1 clase)  — en una T4, del orden de 1-3 h según el tope por origen
dir_1clase = entrenar('1clase')

In [ ]:
# 7. Entrenar la variante B (multiclase, línea base)
dir_multi = entrenar('multiclase')

## 8. Evaluación

- **P (precisión)** = TP / (TP + FP): de lo que el modelo marcó, cuánto era de verdad una bebida.
- **R (recall)** = TP / (TP + FN): de las bebidas reales, cuántas encontró. Para *contar* importan las dos:
  un FP suma uno de más, un FN uno de menos.
- Una detección es TP si su **IoU** (intersección / unión de cajas) con una caja real es ≥ umbral.
- **mAP@0.5**: área bajo la curva precisión-recall (AP) con IoU ≥ 0.5, promediada entre clases.
- **mAP@0.5:0.95**: promedio de los AP con IoU = 0.50, 0.55, …, 0.95: además exige cajas *bien ajustadas*.
- **Error de conteo (MAE)**: |detectadas − reales| promedio por imagen, con el umbral de confianza que usa
  el sistema (`UMBRAL_DET = 0.25`). Es la métrica que le importa al inventario.

In [ ]:
# 8. mAP, P y R en val y test (+ matriz de confusión en multiclase)
metricas = {}
for variante, carpeta in (('1clase', dir_1clase), ('multiclase', dir_multi)):
    modelo = YOLO(str(carpeta / 'weights' / 'best.pt'))
    for split in ('val', 'test'):
        m = modelo.val(data=str(RAIZ / CARPETA[variante] / 'data.yaml'), split=split, imgsz=HIPER['imgsz'],
                       batch=32, device=0, plots=True, project=str(RUNS), name=f'eval_{variante}_{split}',
                       exist_ok=True, verbose=False)
        metricas[f'{variante}_{split}'] = {'mAP50': round(float(m.box.map50), 4), 'mAP50_95': round(float(m.box.map), 4),
                                          'P': round(float(m.box.mp), 4), 'R': round(float(m.box.mr), 4)}
        if variante == 'multiclase':
            metricas[f'{variante}_{split}']['mAP50_95_por_clase'] = {
                m.names[int(c)]: round(float(v), 4) for c, v in zip(m.box.ap_class_index, m.box.maps[m.box.ap_class_index])}
print(json.dumps(metricas, indent=2, ensure_ascii=False))

In [ ]:
# 9. Matriz de confusión de la variante B (test) y curvas de entrenamiento
from IPython.display import Image, display
display(Image(str(RUNS / 'eval_multiclase_test' / 'confusion_matrix_normalized.png'), width=650))
for carpeta in (dir_1clase, dir_multi):
    display(Image(str(carpeta / 'results.png'), width=900))

In [ ]:
# 10. Error de conteo por imagen en test (umbral 0.25), total y por dataset de origen
import pandas as pd

def conteo_test(variante, carpeta, conf=0.25):
    modelo = YOLO(str(carpeta / 'weights' / 'best.pt'))
    base = RAIZ / CARPETA[variante]
    filas = []
    # la carpeta y no una lista de rutas: con una lista, Ultralytics carga todas las imágenes en RAM de una vez
    for r in modelo.predict(str(base / 'images' / 'test'), conf=conf, imgsz=HIPER['imgsz'], device=0,
                            stream=True, verbose=False, max_det=300):
        p = Path(r.path)
        lbl = base / 'labels' / 'test' / (p.stem + '.txt')
        reales = [int(float(l.split()[0])) for l in lbl.read_text().splitlines() if l.strip()] if lbl.exists() else []
        pred = [int(k) for k in r.boxes.cls.cpu().numpy()]
        fila = {'imagen': p.name, 'origen': p.name.split('__')[0], 'n_real': len(reales), 'n_det': len(pred)}
        if variante == 'multiclase':
            fila['error_por_clase_abs'] = sum(abs(pred.count(k) - reales.count(k)) for k in range(8))
        filas.append(fila)
    df = pd.DataFrame(filas)
    df['error'] = df.n_det - df.n_real
    df.to_csv(RUNS / f'conteo_test_{variante}.csv', index=False)
    def resumen(g):
        return pd.Series({'imagenes': len(g), 'MAE': g.error.abs().mean(), 'exacto_%': 100 * (g.error == 0).mean(),
                          'sesgo': g.error.mean()})
    tabla = pd.concat([resumen(df).rename('TOTAL').to_frame().T, df.groupby('origen').apply(resumen)])
    print(variante); print(tabla.round(3).to_string())
    return tabla

conteos = {v: conteo_test(v, c) for v, c in (('1clase', dir_1clase), ('multiclase', dir_multi))}

In [ ]:
# 11. Empaquetar resultados, copiarlos a Drive y descargarlos
SALIDA = Path('/content/resultados_detector')
shutil.rmtree(SALIDA, ignore_errors=True)
SALIDA.mkdir()
for variante, carpeta in (('1clase', dir_1clase), ('multiclase', dir_multi)):
    shutil.copy(carpeta / 'weights' / 'best.pt', SALIDA / f'detector_{variante}.pt')
    shutil.copytree(carpeta, SALIDA / 'runs' / variante, ignore=shutil.ignore_patterns('last.pt'))
for split_dir in RUNS.glob('eval_*'):
    shutil.copytree(split_dir, SALIDA / 'runs' / split_dir.name)
for csv in RUNS.glob('conteo_test_*.csv'):
    shutil.copy(csv, SALIDA)
(SALIDA / 'metricas_colab.json').write_text(json.dumps(metricas, indent=2, ensure_ascii=False))

archivo = shutil.make_archive('/content/resultados_detector', 'zip', SALIDA)
print(archivo, f'{Path(archivo).stat().st_size / 1e6:.0f} MB')
if ORIGEN == 'drive':
    shutil.copy(archivo, CARPETA_DRIVE / 'resultados_detector.zip')
    print('Copia en Drive:', CARPETA_DRIVE / 'resultados_detector.zip')
from google.colab import files
files.download(archivo)

## 12. En la PC, con `resultados_detector.zip` descargado

Descomprimirlo en una carpeta temporal y, en el **repo principal**:

1. `detector_1clase.pt` y `detector_multiclase.pt` → `modelos/detector/`
   (es lo que carga por defecto `src/vision/detector.py`).
2. `runs/1clase` → `runs/detector/colab_1clase`, `runs/multiclase` → `runs/detector/colab_multiclase`,
   `runs/eval_*` → `runs/detector/colab/`.
3. `metricas_colab.json` y `conteo_test_*.csv` → `runs/detector/colab/`.
4. Verificar en la PC:
   ```
   python -m pytest tests/test_detector.py
   python scripts/detector_evaluar.py todo --variante 1clase
   python scripts/detector_evaluar.py todo --variante multiclase
   tensorboard --logdir runs/detector
   ```
5. Abrir `notebooks/04_detector.ipynb` y volver a ejecutarlo: lee esos archivos y muestra curvas, métricas,
   conteo y la comparación con la línea base COCO.